## Evaluation pipeline for both models

Here we present the quantitative evaluation pipeline to compare the two versions of the EoMT semantic segmentation model on the Cityscapes validation set.

The objective is to perform a fair comparison between:

- the EoMT model trained on the Cityscapes dataset.
- the EoMT model trained on the COCO dataset, whose predictions were mapped to the Cityscapes semantic label space.

We perform the evaluation using the predicted segmentation masks previously generated and saved as .png files for both models. The ground truth annotations are taken from the official Cityscapes validation set.

The following quantitative metrics are computed:

- Intersection over Union (IoU) for each class
- mean Intersection over Union (mIoU)
- pixel accuracy

This allows us to quantitatively analyze the effect of dataset alignment on segmentation segmentation task. In particular, it highlights the difference between a model trained directly on Cityscapes and a model trained on a more general COCO dataset. 

*The IoU logic is adapted from eval_iou.py*

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm


NUM_CLASSES = 19
IGNORE_INDEX = 255

CLASS_NAMES = [
    "road", "sidewalk", "building", "wall", "fence", "pole",
    "traffic light", "traffic sign", "vegetation", "terrain",
    "sky", "person", "rider", "car", "truck", "bus",
    "train", "motorcycle", "bicycle"
]

EVAL_CLASSES = [
    0,   # road
    1,   # sidewalk
    2,   # building
    3,   # wall
    4,   # fence
    6,   # traffic light
    7,   # traffic sign
    8,   # vegetation
    10,  # sky
    11,  # person
    13,  # car
    14,  # truck
    15,  # bus
    16,  # train
    17,  # motorcycle
    18,  # bicycle
]

EVAL_CLASS_NAMES = [CLASS_NAMES[i] for i in EVAL_CLASSES]

print(EVAL_CLASS_NAMES)

['road', 'sidewalk', 'building', 'wall', 'fence', 'traffic light', 'traffic sign', 'vegetation', 'sky', 'person', 'car', 'truck', 'bus', 'train', 'motorcycle', 'bicycle']


In [2]:
ground_truth = "../data/datasets_unzip/gtFine/val"
gt_trainId = "../data/datasets_unzip/gtFine_trainIds/val"

city_pred = "../data/eomt_valset_predictions/cityscapes_model/png/content/drive/MyDrive/eomt_valset_predictions/cityscapes_model/png"
coco_pred = "../data/eomt_valset_predictions/coco_model/png/content/drive/MyDrive/eomt_valset_predictions/png"

---

## Limit GT to overlap classes

Since both models are evaluated on the common limited label space, we also need to correspondingly limit the ground-truth labels.

In [3]:
def keep_only_eval_classes(mask, eval_classes):
    filtered = np.full(mask.shape, IGNORE_INDEX, dtype=np.uint8)

    for cls in eval_classes:
        filtered[mask == cls] = cls

    return filtered

Compute the Confusion Matrix. It records how many pixels of each ground-truth class are predicted as each possible class by the model.

In [4]:
def load_mask(path):
    return np.array(Image.open(path), dtype=np.int64)


def confusion_matrix(gt, pred, num_classes=19):
    gt = gt.astype(np.int64)
    pred = pred.astype(np.int64)
    
    valid_mask = (
        (gt != IGNORE_INDEX) &
        (pred != IGNORE_INDEX) &
        (gt >= 0) &
        (gt < num_classes) &
        (pred >= 0) &
        (pred < num_classes)
)

    hist = np.bincount(
        num_classes * gt[valid_mask] + pred[valid_mask],
        minlength=num_classes ** 2
    ).reshape(num_classes, num_classes)

    return hist

The following function evaluates a set of prediction masks against the corresponding Cityscapes ground-truth masks and computes the final semantic segmentation metrics.

In [7]:
def eval_pred(gt_paths, pred_dir):
    total_hist = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.float64)

    missing_predictions = []

    for gt_path in tqdm(gt_paths):
        gt_filename = os.path.basename(gt_path)

        pred_filename = gt_filename.replace(
            "_gtFine_labelTrainIds.png",
            "_predTrainIds.png"
        )

        pred_path = os.path.join(pred_dir, pred_filename)

        if not os.path.exists(pred_path):
            missing_predictions.append(pred_path)
            continue

        gt = load_mask(gt_path)
        gt = keep_only_eval_classes(gt, EVAL_CLASSES)

        pred = load_mask(pred_path)
        pred = keep_only_eval_classes(pred, EVAL_CLASSES)

        if gt.shape != pred.shape:
            raise ValueError(
                f"Shape mismatch:\n"
                f"GT: {gt_path} {gt.shape}\n"
                f"Prediction: {pred_path} {pred.shape}"
            )

        total_hist += confusion_matrix(gt, pred, NUM_CLASSES)

    intersection = np.diag(total_hist)
    union = total_hist.sum(axis=1) + total_hist.sum(axis=0) - intersection

    class_iou = intersection / np.maximum(union, 1)

    mean_iou = np.mean(class_iou[EVAL_CLASSES])

    pixel_accuracy = intersection[EVAL_CLASSES].sum() / np.maximum(
        total_hist[EVAL_CLASSES, :].sum(),
        1
    )

    return {
        "class_iou": class_iou,
        "mean_iou": mean_iou,
        "pixel_accuracy": pixel_accuracy,
        "confusion_matrix": total_hist,
        "missing_predictions": missing_predictions
    }

---

## Run both models

In [9]:
gt_train_root = "../data/datasets_unzip/gtFine_trainIds/val"

gt_train_paths = sorted(glob.glob(
    os.path.join(
        gt_train_root,
        "*",
        "*_gtFine_labelTrainIds.png"
    )
))

print("Number of GT trainId masks:", len(gt_train_paths))
print(gt_train_paths[0])

Number of GT trainId masks: 500
../data/datasets_unzip/gtFine_trainIds/val/frankfurt/frankfurt_000000_000294_gtFine_labelTrainIds.png


In [10]:
city_results = eval_pred(gt_train_paths, city_pred)
coco_results = eval_pred(gt_train_paths, coco_pred)

  0%|          | 0/500 [00:00<?, ?it/s]

100%|██████████| 500/500 [00:38<00:00, 12.93it/s]


Check if both models were evaluate correctly:

In [12]:
print("Missing Cityscapes predictions:", len(city_results["missing_predictions"]))
print("Missing COCO predictions:", len(coco_results["missing_predictions"]))

print(city_results["missing_predictions"][:5])
print(coco_results["missing_predictions"][:5])

Missing Cityscapes predictions: 0
Missing COCO predictions: 0
[]
[]


---

## Final metrics

In [13]:
summary_df = pd.DataFrame({
    "Model": [
        "Cityscapes-trained EoMT",
        "COCO-trained EoMT mapped"
    ],
    "mIoU (%)": [
        city_results["mean_iou"] * 100,
        coco_results["mean_iou"] * 100
    ],
    "Pixel Accuracy (%)": [
        city_results["pixel_accuracy"] * 100,
        coco_results["pixel_accuracy"] * 100
    ]
})

summary_df

,Model,mIoU (%),Pixel Accuracy (%)
0,Cityscapes-trained EoMT,85.588350,97.526996
1,COCO-trained EoMT mapped,61.388808,93.234531


**Cityscapes-trained EoMT model**

The result confirms that GT conversion was made correctly and confusion matrix implementation is correct.

**COCO-trained EoMT model**

The mIoU value is substantially worse, but still surprisingly decent given that it was not trained on Cityscapes and we mapped class spaces. As we expected, the *train* class failed completely. 

The pixel accuracy remains high because stuff classes like *sky, building, vegetation* occupy huge number of pixels.

In [15]:
per_class_iou_df = pd.DataFrame({
    "Class": CLASS_NAMES,
    "Cityscapes-trained IoU (%)": city_results["class_iou"] * 100,
    "COCO-trained mapped IoU (%)": coco_results["class_iou"] * 100
})

per_class_iou_df.iloc[EVAL_CLASSES]

,Class,Cityscapes-trained IoU (%),COCO-trained mapped IoU (%)
0,road,98.505228,94.464253
1,sidewalk,89.230818,67.037439
2,building,95.014607,88.613581
3,wall,67.135802,52.693236
4,fence,67.087469,49.602003
6,traffic light,79.781859,46.971742
7,traffic sign,84.476685,1.854360
8,vegetation,94.673415,89.633106
10,sky,95.799877,89.488076
11,person,87.721021,77.596931


Notice that COCO-trained model struggles on classes like **wall, fence** and especially on **traffic sign**.

- **traffic sign**: the model struggles to predict it because of weak classes overlap. Recall that we matched:

    - 11 -> 7   # stop sign -> traffic sign

Traffic sign is much wider concept than the stop sign. We kept this match because of heuristic assumption. For human intuition is it natural to consider stop sign as a traffic sign, so we decided to experiment on the model.

Notice that classes **road, building, vegetation, sky** transfer very well. They are large and clearly appear in the images and dominate the scenes.

---

## Conclusion

The Cityscapes-trained model achieved a mean IoU of 85.6% and a pixel accuracy of 97.5%, while the COCO-trained model achieved a mean IoU of 61.4% and a pixel accuracy of 93.2%.

The results demonstrate that the Cityscapes-trained model significantly outperforms the COCO-trained model when evaluated on Cityscapes scenes, which is expected given that it was trained on the target domain.

The following work for this pipeline would be an attempt to fine-tune the COCO-trained model on the Cityscapes dataset and evaluate how much its performance can be improved.